# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/xtallet/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/xtallet/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '0cb503'. Skipping!
Property 'summary' already exists in node '039aeb'. Skipping!
Property 'summary' already exists in node '498903'. Skipping!
Property 'summary' already exists in node 'bcd99a'. Skipping!
Property 'summary' already exists in node 'f19c7c'. Skipping!
Property 'summary' already exists in node '51fdf4'. Skipping!
Property 'summary' already exists in node '246a4f'. Skipping!
Property 'summary' already exists in node '1f67e2'. Skipping!
Property 'summary' already exists in node '011b2d'. Skipping!
Property 'summary' already exists in node '2be0f6'. Skipping!
Property 'summary' already exists in node 'e734c5'. Skipping!
Property 'summary' already exists in node '4f4ff2'. Skipping!
Property 'summary' already exists in node '966ff5'. Skipping!
Property 'summary' already exists in node '883a99'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '0cb503'. Skipping!
Property 'summary_embedding' already exists in node '498903'. Skipping!
Property 'summary_embedding' already exists in node '1f67e2'. Skipping!
Property 'summary_embedding' already exists in node 'bcd99a'. Skipping!
Property 'summary_embedding' already exists in node '4f4ff2'. Skipping!
Property 'summary_embedding' already exists in node '51fdf4'. Skipping!
Property 'summary_embedding' already exists in node '039aeb'. Skipping!
Property 'summary_embedding' already exists in node 'e734c5'. Skipping!
Property 'summary_embedding' already exists in node 'f19c7c'. Skipping!
Property 'summary_embedding' already exists in node '2be0f6'. Skipping!
Property 'summary_embedding' already exists in node '011b2d'. Skipping!
Property 'summary_embedding' already exists in node '246a4f'. Skipping!
Property 'summary_embedding' already exists in node '966ff5'. Skipping!
Property 'summary_embedding' already exists in node '883a99'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 478)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 478)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5), # 🏗️ XTALLET Notes - Generates simple and direct questions
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25), # 🏗️ XTALLET Notes - Generates more complex and abstract questions
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25), # 🏗️ XTALLET Notes - Generates specific and complex questions
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### ✅ Answer:

- SingleHopSpecificQuerySynthesizer :
  It generates simple and direct questions that can be answered by looking at a single piece of information in the text.<br>
  It makes easy and direct questions.<br>
  These questions will represent the 50% of the total list of queries.

- MultiHopAbstractQuerySynthesizer :
  Generates more complex and general questions that require combining information from different parts of the text and thinking more abstractly.<br>
  It makes broad and challenging questions.<br>
  These questions will represent the 25% of the total list of queries.

- MultiHopSpecificQuerySynthesizer : 
  Generates specific but complex questions that require searching and connecting information from multiple parts of the text to give a concrete answer.<br>
  It makes detailed questions that require gathering information from different parts of the text.<br>
  These questions will represent the 25% of the total list of queries.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is Title IV in relation to academic progr...,"[Chapter 1 Academic Years, Academic Calendars,...",The context does not provide a specific defini...,single_hop_specifc_query_synthesizer
1,What is 34 CFR 668.3(a)?,[Regulatory Citations Academic year minimums: ...,Regulatory Citations Academic year minimums: 3...,single_hop_specifc_query_synthesizer
2,Volume 8 is it like a standard term or no?,[Inclusion of Clinical Work in a Standard Term...,The context explains that terms including clin...,single_hop_specifc_query_synthesizer
3,Is the FWS program considered a term or non-te...,[Non-Term Characteristics A program that measu...,The FWS program is not considered a term progr...,single_hop_specifc_query_synthesizer
4,What is a Direct Loan in the context of federa...,[both the credit or clock hours and the weeks ...,A Direct Loan is a type of federal student loa...,single_hop_specifc_query_synthesizer
5,Considering the requirements for establishing ...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...","The minimum weeks of instructional time, manda...",multi_hop_abstract_query_synthesizer
6,disbursements and 34 CFR 668.3(b) how do they ...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The context explains that academic year requir...,multi_hop_abstract_query_synthesizer
7,How do the disbursements relate to the require...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Disbursements are connected to the requirement...,multi_hop_abstract_query_synthesizer
8,How do Volume 8 and Volume 7 relate to the inc...,[<1-hop>\n\nboth the credit or clock hours and...,Volume 8 discusses the inclusion of clinical w...,multi_hop_specific_query_synthesizer
9,Whre do I find the info in Appendix A and B ab...,[<1-hop>\n\nDisbursement Timing in Subscriptio...,"The detailed guidance on disbursement rules, i...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'b50cac'. Skipping!
Property 'summary' already exists in node '67bae1'. Skipping!
Property 'summary' already exists in node 'e3aa1a'. Skipping!
Property 'summary' already exists in node '5414aa'. Skipping!
Property 'summary' already exists in node 'f5f0f9'. Skipping!
Property 'summary' already exists in node 'c05816'. Skipping!
Property 'summary' already exists in node '0adacf'. Skipping!
Property 'summary' already exists in node '8b7207'. Skipping!
Property 'summary' already exists in node 'c45092'. Skipping!
Property 'summary' already exists in node '5b0f21'. Skipping!
Property 'summary' already exists in node '26d9c0'. Skipping!
Property 'summary' already exists in node '0790c4'. Skipping!
Property 'summary' already exists in node '7f56e8'. Skipping!
Property 'summary' already exists in node 'ae966c'. Skipping!


...

Property 'summary_embedding' already exists in node '67bae1'. Skipping!
Property 'summary_embedding' already exists in node 'f5f0f9'. Skipping!
Property 'summary_embedding' already exists in node '5b0f21'. Skipping!
Property 'summary_embedding' already exists in node 'e3aa1a'. Skipping!
Property 'summary_embedding' already exists in node '26d9c0'. Skipping!
Property 'summary_embedding' already exists in node '5414aa'. Skipping!
Property 'summary_embedding' already exists in node 'b50cac'. Skipping!
Property 'summary_embedding' already exists in node '0790c4'. Skipping!
Property 'summary_embedding' already exists in node 'c05816'. Skipping!
Property 'summary_embedding' already exists in node '7f56e8'. Skipping!
Property 'summary_embedding' already exists in node 'c45092'. Skipping!
Property 'summary_embedding' already exists in node '0adacf'. Skipping!
Property 'summary_embedding' already exists in node 'ae966c'. Skipping!
Property 'summary_embedding' already exists in node '8b7207'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is the significance of Title IV in relati...,"[Chapter 1 Academic Years, Academic Calendars,...",The context provided does not explicitly defin...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(b) mean in terms of wee...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(b) refers to the weeks of instruc...,single_hop_specifc_query_synthesizer
2,"According to Chapter 3, how are clinical work ...",[Inclusion of Clinical Work in a Standard Term...,Inclusion of clinical work in a standard term ...,single_hop_specifc_query_synthesizer
3,Federal Work Study is it part of the payment p...,[Non-Term Characteristics A program that measu...,The Federal Work-Study (FWS) Program is an exc...,single_hop_specifc_query_synthesizer
4,How does policy compliance for term lengths an...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Policy compliance for term lengths and academi...,multi_hop_abstract_query_synthesizer
5,How does the impact of term length variations ...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in standard ter...,multi_hop_abstract_query_synthesizer
6,How do the disbursement timing requirements fo...,[<1-hop>\n\nboth the credit or clock hours and...,Disbursement timing requirements for federal s...,multi_hop_abstract_query_synthesizer
7,How does credit hour allocation for clinical e...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The context explains that clinical work includ...,multi_hop_abstract_query_synthesizer
8,How does Volume 8 influence disbursement timin...,[<1-hop>\n\nboth the credit or clock hours and...,Volume 8 provides guidance on the impact of ac...,multi_hop_specific_query_synthesizer
9,How do Volume 2 and Volume 8 relate to academi...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Volume 2 discusses the academic year requireme...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available mentioned in the context are:\n\n- Direct Subsidized Loans  \n- Direct Unsubsidized Loans  \n- Direct PLUS Loans (including student Federal PLUS Loans and parent Direct PLUS Loans)  \n- Subsidized and Unsubsidized Federal Stafford Loans (made under the FFEL Program before July 1, 2010)  \n- Federal SLS Loans  \n- Federal PLUS Loans (made under the FFEL Program before July 1, 2010)'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

##### ✅ Answer:

- QA_EVALUATOR :
  Evaluates the correctness of the model's answer. It checks wheter the generated response accurately answers the user's question based on the reference answer (the answer from synthetic dataset).

- LABELED_HELPFULNESS_EVALUATOR :
  Evaluates the helpfulness of the response. It determines if the answer is useful and supportive to the user, taking into account the correct reference answer (the answer from synthetic dataset).

- EMPATHY_EVALUATOR :
  Evaluates the empathy in the response. It checks whether the answer is empathetic and makes the user feel heard and understood.
   

## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'drab-coach-8' at:
https://smith.langchain.com/o/c21c7da9-346c-4a28-b19c-bb6d2faffe60/datasets/6efe3237-76c6-452f-a1ad-a3d39ea515b5/compare?selectedSessions=b9dfeb69-8e1f-4246-a76c-7a999098c68c




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,Considering the information in Volume 2 regard...,"Based on the provided context, the definitions...",None,Volume 2 explains that an academic year must i...,1,0,0,13.455460,6e238a40-6289-4499-a5c8-ba918746c591,9cac7087-ff2e-4531-add8-4a45c62128fe
1,How does Volume 8 explain the disbursement tim...,"Based on the provided context, Volume 8 discus...",None,Volume 8 details that for subscription-based p...,1,0,0,8.769606,ac2d5d9d-add9-4acb-ae84-c0adff69eca2,4c15dbe4-6ef5-47c8-890b-dfc0db345bf1
2,How do Volume 2 and Volume 8 relate to academi...,I don't know.,None,Volume 2 discusses the academic year requireme...,0,0,0,3.026888,0ceb3b2f-be9e-40b3-8a30-4427f96af647,2baf5341-276d-4f4b-bcf1-32d24ca871db
3,How does Volume 8 influence disbursement timin...,Volume 8 provides additional guidance on certa...,None,Volume 8 provides guidance on the impact of ac...,1,0,0,2.617524,8c05f4a9-324c-40ac-b9d9-c5a48100d463,6091c390-40e2-4b06-a36b-264a779b1525
4,How does credit hour allocation for clinical e...,I don't know.,None,The context explains that clinical work includ...,0,0,0,1.023618,e5d2152f-4e4b-4179-9815-7cf3533a3c46,512668fe-40c4-492d-8c81-0cb9e53d6f95
5,How do the disbursement timing requirements fo...,Based on the provided context:\n\n**For clock-...,None,Disbursement timing requirements for federal s...,1,0,0,15.903174,7bf849e6-dfcc-45d7-a8dc-6b2c1ba21602,21c90934-1c21-449b-bddf-c6cb37e0e47e
6,How does the impact of term length variations ...,"Based on the provided context, the classificat...",None,The inclusion of clinical work in standard ter...,1,1,0,11.000764,f4971c7a-1eec-4c7a-b7ed-e0c1c6d1c0bc,ba0f4757-f41b-4d38-bd9c-3ac489d3d3ea
7,How does policy compliance for term lengths an...,Policy compliance for term lengths and academi...,None,Policy compliance for term lengths and academi...,1,1,0,7.332676,8a95485a-e327-4df4-9c41-75b21d9c7536,25816525-c9b4-4827-8bd6-cc5d11ba1b62
8,Federal Work Study is it part of the payment p...,Federal Work-Study (FWS) is not part of the pa...,None,The Federal Work-Study (FWS) Program is an exc...,1,1,0,1.707202,ff54afb2-52fe-41b7-a12d-63ff16b4ab16,c672b70f-556a-4503-b58f-1ee30132cadd
9,"According to Chapter 3, how are clinical work ...","According to the context from Chapter 3, clini...",None,Inclusion of clinical work in a standard term ...,1,1,0,4.771743,5c3da323-50c7-4528-8d90-ad6e4800aaad,032a164e-e9a4-42a3-8cd7-a7be59aa3f34


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

##### ✅ Answer:

Modifying the chunk size can signigicantly impact on :
 - Retrieval Quality :<br> 
   Smaller chunks provide more precise and focused information   retrieval, but may miss broader context.<br>
   Larger chunks : Capture more comprehensive context but may include irrelevant information that dilutes the answer quality.

 - Context Window Efficiency :<br>
   Smaller chunks fit better within the LLM's context window, allowing more relevant chunks to be included in the prompt.<br>
   Larger chunks consume more of the context window, potentially limiting the number of different sources that can be referenced.

 - Semantic Search Accuracy :<br>
   Smaller chunks create more granular embeddings, making it easier to find highly specific information that directly answers the question.<br>
   Larger chunks may have broader semantic meaning but could be less precise for specific queries.

 - Information Completeness :<br>
   Smaller chunks might fragment important information that spans across chunk boundaries, leading to incomplete answers.<br>
   Larger chunks are more likely to contain complete information but may include unnecessary details.

 - Processing Speed :<br>
   Smaller chunks generally result in faster embedding generation and retrieval due to their size.
   Larger chunks require more computational resources but may reduce the number of chunks to process.<br>

   After modify the `chunk_size` and `chunk_overlap` it would affect how the system retrieves and processes the information, potentially improving or degrading performance depending on the specific use case.

In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

##### ✅ Answer:

Modifying the embedding model can significantly impact on :
 
 - Semantic Understanding Quality:<br>
   Different embedding models have varying capabilities in understanding semantic relationships, context, and nuances in text.<br>
   More advanced models (like text-embedding-3-large vs text-embedding-3-small) can capture more sophisticated semantic patterns and relationships.<br>
   Better semantic understanding leads to more accurate retrieval of relevant context for user queries.

 - Retrieval Accuracy :<br>
   Embedding quality directly affects how well the vector similarity search works.<br>
   Superior embedding models can better distinguish between similar but different concepts, reducing false positives and improving precision.<br>
   Poor embeddings may lead to retrieving irrelevant chunks or missing important context.

 - Multilingual and Domain Performance :<br>
   Different models may perform better on specific languages, domains, or types of content.<br>
   Specialized models might handle technical jargon, industry-specific terms, or cultural context better than general-purpose models.

 - Dimensionality and Representation :<br>
   Different embedding dimensions (e.g., 1536 for text-embedding-3-large vs 1024 for text-embedding-3-small) can capture more or less information.<br>
   Higher dimensionality often means richer representations but may require more computational resources.

 - Training Data and Knowledge Cutoff :<br>
   Newer models may have been trained on more recent data, better understanding current events, terminology, or concepts.<br>
   Model updates can improve performance on specific types of queries or content.

In [35]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [36]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [37]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [38]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question. Based on the information in the context, there are several types of loans available to students and their parents to help cover the Cost of Attendance (COA):\n\n1. **Direct Subsidized Loans** – These are loans for students with financial need, where the amount borrowed cannot exceed the student’s financial need (COA minus other aid). Interest is generally paid by the government while the student is in school.\n\n2. **Direct Unsubsidized Loans** – Available to both dependent and independent students, these loans do not require demonstration of financial need. Interest accrues while the student is in school.\n\n3. **Direct PLUS Loans** – These loans are available to the parents of dependent students (Direct PLUS Loans), or to graduate/professional students (student Direct PLUS Loans). They can cover up to the full COA minus other financial aid and do not have fixed loan limits. Eligibility requirements need to be met, and if the parent is ineligible, the dep

Finally, we can evaluate the new chain on the same test set!

In [39]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'upbeat-poison-90' at:
https://smith.langchain.com/o/c21c7da9-346c-4a28-b19c-bb6d2faffe60/datasets/6efe3237-76c6-452f-a1ad-a3d39ea515b5/compare?selectedSessions=fe20efcc-0300-49fa-9580-d6554b88ed44




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,Considering the information in Volume 2 regard...,Thank you for your thoughtful question—it's cl...,None,Volume 2 explains that an academic year must i...,1,1,1,8.600963,6e238a40-6289-4499-a5c8-ba918746c591,dacfbb54-6076-4714-b4f5-e2b3949908db
1,How does Volume 8 explain the disbursement tim...,Thank you for your thoughtful question. From t...,None,Volume 8 details that for subscription-based p...,1,0,1,4.807701,ac2d5d9d-add9-4acb-ae84-c0adff69eca2,2477bfd8-3ff9-40a6-bf5d-2c18eaf1f264
2,How do Volume 2 and Volume 8 relate to academi...,Thank you for your thoughtful question. From t...,None,Volume 2 discusses the academic year requireme...,0,0,1,4.699745,0ceb3b2f-be9e-40b3-8a30-4427f96af647,cd6aca77-3a57-45c4-8f19-278772086fd6
3,How does Volume 8 influence disbursement timin...,Thank you for your thoughtful question. Based ...,None,Volume 8 provides guidance on the impact of ac...,1,1,1,8.787280,8c05f4a9-324c-40ac-b9d9-c5a48100d463,51c988b4-1a2e-4c69-850c-ca35a431a341
4,How does credit hour allocation for clinical e...,Thank you for your thoughtful question. Based ...,None,The context explains that clinical work includ...,1,1,1,6.420362,e5d2152f-4e4b-4179-9815-7cf3533a3c46,caf35f8c-1a2e-4503-93eb-0343f35a0b17
5,How do the disbursement timing requirements fo...,Thank you for your thoughtful question—it's cl...,None,Disbursement timing requirements for federal s...,1,0,1,7.832551,7bf849e6-dfcc-45d7-a8dc-6b2c1ba21602,a6e638c9-0cf9-4003-86e4-a6c6530a4251
6,How does the impact of term length variations ...,Thank you for your thoughtful question. I can ...,None,The inclusion of clinical work in standard ter...,1,1,1,10.416074,f4971c7a-1eec-4c7a-b7ed-e0c1c6d1c0bc,82494e2f-b1cd-4a5b-bc05-c3a7c8db5963
7,How does policy compliance for term lengths an...,Thank you for your thoughtful question about s...,None,Policy compliance for term lengths and academi...,1,1,1,6.100510,8a95485a-e327-4df4-9c41-75b21d9c7536,dbfc3607-d37e-4aba-94d4-f6409130b492
8,Federal Work Study is it part of the payment p...,Thank you for your question about Federal Work...,None,The Federal Work-Study (FWS) Program is an exc...,1,1,1,4.598314,ff54afb2-52fe-41b7-a12d-63ff16b4ab16,3971e434-41cb-4ad1-842c-e7bc3f159ea8
9,"According to Chapter 3, how are clinical work ...",Thank you for your thoughtful question. Based ...,None,Inclusion of clinical work in a standard term ...,1,1,1,4.971353,5c3da323-50c7-4528-8d90-ad6e4800aaad,6a316523-0667-4a05-853a-d9d33e398326


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

##### ✅ Answer:

First Evaluation Config : 
 - Prompt : RAG_PROMPT
 - Embedding Model : it is not specified so by default it should be text-embedding-3-amsll
 - Chunking : chunk_size : 500 - chunk_overlap : 50 

Second Evaluation Config : 
 - Prompt : EMPATHY_RAG_PROMPT
 - Embedding Model : text-embedding-3-large
 - Chunking : chunk_size : 1000 - chunk_overlap : 50

These are my observations on the three types of evaluations we have tested :<br>

#### Correctness
The number of answers marked as correct increased in the second evaluation.<br>
Probable reasons :<br>

- Better Embedding model improved the retrieval of relevant context, so the model had more accurate information to answer the questions.<br>
- Larger chunk size meant each retrieved chunk contained more context, reducing the chance of missing key information needed for a correct answer.<br>
- The prompt change to an empathy-focused version did not negatively affect correctness, as the model was still instructed to answer based on context.

#### Helpfulness
Helpfulness scores generally improved or remained high for correct answers.<br>
Probable reasons :<br>

 - More relevant and complete context (due to better embeddings and larger chunks) allowed the model to provide more useful and informative answers.
 - The empathy prompt may have encouraged the model to give more supportive and user-focused responses, which can be perceived as more helpful.

#### Empathy
Empathy scores increased significantly in the second evaluation.<br>
Probable reasons :<br>

- The empathy-focused prompt explicitly instructed the model to be empathetic and make the user feel heard.<br>
- As a result, the model’s responses included more empathetic language, which was recognized by the evaluator.<br>

#### Screenshots
 - This screenshot shows both evaluations comparisson
<img src="screenshots/activity3_scrn1.png" alt="Loan Synthetic Data" width="1200"/>

 - This screenshot shows the details about the Evaluation 1 :
 <img src="screenshots/activity3_scrn2.png" alt="Loan Synthetic Data" width="1200"/>

 - This screenshot shows the details about the Evaluation 2 :
 <img src="screenshots/activity3_scrn3.png" alt="Loan Synthetic Data" width="1200"/>
